# MISTRAL 7B- Fine Tuning

In [1]:
from collections import Counter
import json

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []

    for key, value in data.items():
        tweet_text = value['tweet']
        votos = value['labels_task1']
        
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue 
            
        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])

    return X, y, ids



ruta = 'dataset_task1_exist2025/training.json' 
try:
    X_train, y_train, ids_train = cargar_datos_entrenamiento(ruta)
    
    print(f"Procesados {len(X_train)} tweets correctamente.")
    print(f"Ejemplo - Texto: {X_train[0][:50]}...")
    print(f"Ejemplo - Label: {y_train[0]}")

except FileNotFoundError:
    print("Error: No se encontró el archivo training.json. Revisa la ruta.")

Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [2]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from trl import SFTTrainer, SFTConfig

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

# Split 80/20 estratificado para evaluar tras el entrenamiento
df_train_split, eval_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"],
)

df_train_split = df_train_split.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

train_dataset = Dataset.from_pandas(df_train_split)
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["text", "label"]]
)

eval_dataset = Dataset.from_pandas(eval_df)
eval_dataset = eval_dataset.remove_columns(
    [col for col in eval_dataset.column_names if col not in ["text", "label"]]
)


model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        max_length=256,
        truncation=True,
        padding="max_length",
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_dataset = eval_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])


compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


sft_config = SFTConfig(
    output_dir="./mistral-sexism-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    max_grad_norm=0.0,
)


print("Precision config -> fp16:", sft_config.fp16, "bf16:", sft_config.bf16)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=sft_config,
)
trainer.train()


PROMPT_TEMPLATE = (
    "Clasifica el siguiente tweet como SEXISTA o NO SEXISTA. "
    "Responde SOLO con YES o NO.\n\n"
    "Tweet: {tweet}\n"
    "Respuesta:"
)

def _to_label(txt):
    t = txt.strip().upper()
    if "YES" in t:
        return "YES"
    if "NO" in t:
        return "NO"
    return "NO"

def predict_labels(texts, batch_size=8, max_new_tokens=4):
    preds = []
    trainer.model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            prompts = [PROMPT_TEMPLATE.format(tweet=t) for t in batch]
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(trainer.model.device)

            outputs = trainer.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            gen_only = outputs[:, inputs["input_ids"].shape[1]:]
            decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
            preds.extend([_to_label(x) for x in decoded])
    return preds

y_true_eval = eval_df["label"].tolist()
y_pred_eval = predict_labels(eval_df["text"].tolist(), batch_size=8)

f1_macro_eval = f1_score(y_true_eval, y_pred_eval, average="macro")
print(f"F1-macro en eval (20%): {f1_macro_eval:.4f}")
print(classification_report(y_true_eval, y_pred_eval, digits=4))

c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 1213/1213 [00:00<00:00, 16829.33 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
W0304 19:17:37.176000 30040 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|▏         | 4/291 [00:02<06:27,  1.35s/it, Materializing param=model.layers.0.mlp.down_proj.weight]  c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:12<00:00, 24.00it/s, Materializi

Precision config -> fp16: False bf16: False


Truncating eval dataset: 100%|██████████| 1213/1213 [00:00<00:00, 348304.97 examples/s]


Epoch,Training Loss,Validation Loss
1,0.756805,0.799739


F1-macro en eval (20%): 0.4226
              precision    recall  f1-score   support

          NO     0.5320    0.7656    0.6277       674
         YES     0.3498    0.1577    0.2174       539

    accuracy                         0.4955      1213
   macro avg     0.4409    0.4616    0.4226      1213
weighted avg     0.4510    0.4955    0.4454      1213

